In [20]:
import pandas as pd
import numpy as np
df = pd.read_csv("../data/new/synthetic_cardiac_100k.csv")

df.shape
df.info()
df.describe(include="all").T
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 16 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   patient_id           100000 non-null  object 
 1   age                  100000 non-null  int64  
 2   sex                  100000 non-null  object 
 3   resting_bp           98507 non-null   float64
 4   cholesterol          98002 non-null   float64
 5   bmi                  98505 non-null   float64
 6   fasting_blood_sugar  100000 non-null  int64  
 7   max_heart_rate       98797 non-null   float64
 8   smoking              100000 non-null  int64  
 9   diabetes             100000 non-null  int64  
 10  physical_activity    100000 non-null  int64  
 11  family_history       100000 non-null  int64  
 12  chest_pain_type      99002 non-null   object 
 13  resting_ecg          99003 non-null   object 
 14  exercise_angina      100000 non-null  int64  
 15  cardiac_risk      

,patient_id,age,sex,resting_bp,cholesterol,bmi,fasting_blood_sugar,max_heart_rate,smoking,diabetes,physical_activity,family_history,chest_pain_type,resting_ecg,exercise_angina,cardiac_risk
0,SYN-075722,49,Female,113.1,194.4,32.0,0,NaN,0,0,0,0,asymptomatic,normal,0,0
1,SYN-080185,69,Male,124.9,233.0,27.9,0,165.3,0,0,1,0,atypical_angina,left_ventricular_hypertrophy,0,0
2,SYN-019865,45,Female,141.6,175.3,29.9,0,198.6,0,0,1,1,typical_angina,normal,0,0
3,SYN-076700,51,Female,140.0,245.9,21.4,0,180.9,1,0,0,0,atypical_angina,normal,0,0
4,SYN-092992,42,Male,112.2,190.7,26.6,0,182.5,1,0,1,1,atypical_angina,normal,0,0


In [21]:

df.duplicated().sum()

np.int64(500)

In [22]:
df = df.drop_duplicates()
df.duplicated().sum()


np.int64(0)

In [27]:
class_balance = df['cardiac_risk'].value_counts().sort_index().rename(index={0: 'Lower pattern', 1: 'Elevated pattern'})

print('Class counts:')
display(class_balance.to_frame('count').assign(percent=lambda x: (100 * x['count'] / x['count'].sum()).round(1)))

Class counts:


,count,percent
cardiac_risk,,
Lower pattern,69687,70.0
Elevated pattern,29813,30.0


In [23]:

valid_ranges = {
    'age': (18, 100), 'resting_bp': (70, 250), 'cholesterol': (80, 600),
    'bmi': (10, 60), 'fasting_blood_sugar': (0, 1), 'max_heart_rate': (50, 250),
}
invalid_report = {}
for column, (lower, upper) in valid_ranges.items():
    invalid_mask = df[column].notna() & ~df[column].between(lower, upper)
    invalid_report[column] = int(invalid_mask.sum())
    df.loc[invalid_mask, column] = np.nan

valid_categories = {
    'sex': {'Female', 'Male'},
    'chest_pain_type': {'non_anginal', 'asymptomatic', 'typical_angina', 'atypical_angina'},
    'resting_ecg': {'normal', 'st_t_abnormality', 'left_ventricular_hypertrophy'},
}
for column, allowed in valid_categories.items():
    invalid_mask = df[column].notna() & ~df[column].isin(allowed)
    invalid_report[column] = int(invalid_mask.sum())
    df.loc[invalid_mask, column] = np.nan

df = df[df['cardiac_risk'].isin([0, 1])].copy()
df['cardiac_risk'] = df['cardiac_risk'].astype(int)

quality_summary = pd.DataFrame({
    'missing_after_validation': df.isna().sum(),
    'missing_percent': (100 * df.isna().mean()).round(2),
})
print(f'Cleaned shape before pipeline imputation: {df.shape}')
print('Invalid values converted to missing:', invalid_report)
display(quality_summary)

Cleaned shape before pipeline imputation: (99500, 16)
Invalid values converted to missing: {'age': 99, 'resting_bp': 99, 'cholesterol': 99, 'bmi': 99, 'fasting_blood_sugar': 0, 'max_heart_rate': 0, 'sex': 0, 'chest_pain_type': 0, 'resting_ecg': 0}


,missing_after_validation,missing_percent
patient_id,0,0.0
age,99,0.1
sex,0,0.0
resting_bp,1588,1.6
cholesterol,2087,2.1
bmi,1591,1.6
fasting_blood_sugar,0,0.0
max_heart_rate,1194,1.2
smoking,0,0.0
diabetes,0,0.0


### Saving the dataset

In [26]:
df.to_csv(
    "../data/processed/cardiac_validated.csv",
    index=False
)